In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from pathlib import Path

In [3]:
print(sys.path)

['/opt/homebrew/Cellar/python@3.11/3.11.8/Frameworks/Python.framework/Versions/3.11/lib/python311.zip', '/opt/homebrew/Cellar/python@3.11/3.11.8/Frameworks/Python.framework/Versions/3.11/lib/python3.11', '/opt/homebrew/Cellar/python@3.11/3.11.8/Frameworks/Python.framework/Versions/3.11/lib/python3.11/lib-dynload', '', '/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/de_etl_examples/.venv/lib/python3.11/site-packages', '/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/de_etl_examples/src', '/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/de_etl_examples/.venv/lib/python3.11/site-packages/setuptools/_vendor', '/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/adm_integration_framework']


In [4]:
root_path = (Path(os.getcwd()).parent.parent / "src").as_posix()
root_path

'/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/adm_integration_framework/src'

In [5]:
sys.path.append(root_path)

In [6]:
from adm.integration.adm_data_transformer import AdmDataTransformer
from adm.integration.adm_base_data_transformer import AdmBaseDataTransformer
from adm.integration.adm_table_writer import AdmTableWriter
from adm.integration.adm_table_reader import AdmTableReader


In [7]:
config_file = "use_cases/p2c_human_nutrition_open_orders/human_nutrition_open_orders_sql.yml"
adm_table_reader = AdmTableReader(config_file)
transformer = AdmBaseDataTransformer(config_file)
transformer2 = AdmDataTransformer(config_file)
writer = AdmTableWriter(config_file)


In [8]:
df_ibm = transformer.apply_transformations("ibmdb2")

INFO:adm.integration.base_integration:Starting transformations for ID: ibmdb2
INFO:adm.integration.base_integration:Initializing DataFrame from SQL file: ../../use_cases/p2c_human_nutrition_open_orders/sql/ibm_open.sql


In [9]:
df_jde_eu = transformer.apply_transformations("jde_eu")

INFO:adm.integration.base_integration:Starting transformations for ID: jde_eu
INFO:adm.integration.base_integration:Initializing DataFrame from SQL file: ../../use_cases/p2c_human_nutrition_open_orders/sql/jde_eu_open.sql


In [10]:
df_u = df_ibm.unionByName(df_jde_eu, allowMissingColumns=True)

In [11]:
spark.conf.set("spark.sql.legacy.charVarcharAsString", True)

In [12]:
writer.write_table("hn_open_orders_sql", df_u)

INFO:adm.integration.base_integration:Number of records written to table sandbox.de_ref_silver_mikesturwoldcontractor.hn_open_orders_sql: 22781


In [ ]:
#display(spark.table("sandbox.de_ref_silver_mikesturwoldcontractor.hn_open_orders_sql").groupBy("source").count().orderBy("source"))

,source,count
0,IBM,3525
1,JDE EU,19256
